# Phase 5 Visualization — Analysis Angle Structure

Visualizations structured by analysis function type (F1–F5) per `04_analysis_angle_structure.md`:

| Type | Question | Chart |
|------|----------|-------|
| F1 | How is property X distributed? | Horizontal bar chart |
| F2 | How does distribution change over time? | Line chart (year × value) |
| F3 | Property X × Property Y counts? | Centered stacked bar (binary) or grouped bar |
| F4 | Distribution of a numeric property? | Histogram + violin |
| F5 | How do values nest in the class hierarchy? | Sunburst (primary) + Sankey |
| PR | Page rank across entities | Node graph |

**Inputs:** `data/50_analysis/all/` (analysis outputs from `50_analysis.ipynb`)
**Outputs:** `data/50_analysis/all/visualizations/` (PNG + PDF + HTML)


In [ ]:
from pathlib import Path
import os, sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    for parent in list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break
os.chdir(repo_root)

ANALYSIS_DIR = Path("data/50_analysis/all")
VIZ_DIR = ANALYSIS_DIR / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# --- Okabe-Ito colorblind-safe palette (visualization-principles.md §1) ---
PALETTE = {
    "blue":       "#0072B2",
    "orange":     "#E69F00",
    "green":      "#009E73",
    "sky":        "#56B4E9",
    "vermillion": "#D55E00",
    "purple":     "#CC79A7",
    "yellow":     "#F0E442",
    "black":      "#000000",
    "gray":       "#999999",  # always: Unknown / Other / neutral
}
PALETTE_SEQ = ["blue", "orange", "green", "sky", "vermillion", "purple", "yellow", "black"]

GENDER_COLOR = {
    "weiblich": PALETTE["blue"],
    "männlich": PALETTE["orange"],
    "female":   PALETTE["blue"],
    "male":     PALETTE["orange"],
    "(unknown)": PALETTE["gray"],
}

# Font constant (visualization-principles.md §2) — set to None for system font
FONT_FAMILY = None
FONT_SIZE_BASE = 12

def apply_font(fig):
    fig.update_layout(font=dict(family=FONT_FAMILY, size=FONT_SIZE_BASE))
    return fig

def save_fig(fig, name: str, html: bool = True) -> None:
    base = VIZ_DIR / name
    fig.write_image(str(base) + ".png", scale=3)
    fig.write_image(str(base) + ".pdf")
    if html:
        fig.write_html(str(base) + ".html")
    print(f"  Saved: {name}.png / .pdf" + (" / .html" if html else ""))

def palette_for(values, unknown_keys=("(unknown)", "")):
    colors = {}
    idx = 0
    for v in values:
        if v in unknown_keys:
            colors[v] = PALETTE["gray"]
        elif v in GENDER_COLOR:
            colors[v] = GENDER_COLOR[v]
        else:
            colors[v] = PALETTE[PALETTE_SEQ[idx % len(PALETTE_SEQ)]]
            idx += 1
    return colors

print(f"VIZ_DIR: {VIZ_DIR.resolve()}")
print("Setup complete.")


## Load Analysis Data

Reads all analysis CSV outputs from `data/50_analysis/all/`.


In [ ]:
def _load(name):
    p = ANALYSIS_DIR / name
    if p.exists():
        return pd.read_csv(p, dtype=str).fillna("")
    print(f"  WARNING: {name} not found — run 50_analysis.ipynb first")
    return pd.DataFrame()

gender_dist      = _load("gender_distribution.csv")
gender_time      = _load("gender_over_time.csv")
occ_dist         = _load("occupation_distribution.csv")
party_dist       = _load("party_distribution.csv")
gender_by_occ    = _load("gender_by_occupation.csv")
gender_by_party  = _load("gender_by_party.csv")
party_by_occ     = _load("party_by_occupation.csv")
age_dist         = _load("age_distribution.csv")

# Numeric conversions
for df, cols in [
    (gender_dist, ["person_count","appearance_count","pct_by_person","pct_by_appearance"]),
    (gender_time, ["person_count","appearance_count"]),
    (occ_dist,    ["person_count","appearance_count","pct_by_person"]),
    (party_dist,  ["person_count","appearance_count","pct_by_person"]),
    (age_dist,    ["appearance_count","unique_persons","mean_age"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

print("Data loaded.")
for name, df in [
    ("gender_dist", gender_dist), ("gender_time", gender_time),
    ("occ_dist", occ_dist), ("party_dist", party_dist),
    ("gender_by_occ", gender_by_occ), ("gender_by_party", gender_by_party),
    ("party_by_occ", party_by_occ), ("age_dist", age_dist),
]:
    print(f"  {name}: {len(df)} rows {'✓' if len(df) > 0 else '⚠ EMPTY'}")


## F1 — Property Distribution (Horizontal Bar Charts)

One chart per property: gender, occupation, party.
Chart type: horizontal bar (sorted descending, Unknown/Other last in gray).
Source: F1 analysis outputs.


In [ ]:
# F1a — Gender distribution
if not gender_dist.empty:
    df = gender_dist.sort_values("person_count", ascending=False).copy()
    unknowns = df[df["value"].isin(["(unknown)", ""])].copy()
    known    = df[~df["value"].isin(["(unknown)", ""])].copy()
    df = pd.concat([known, unknowns], ignore_index=True)
    df = df.iloc[::-1]  # plotly horizontal bar: last row = top of chart

    colors = [GENDER_COLOR.get(v, PALETTE["gray"]) for v in df["value"]]
    total  = int(df["person_count"].sum())

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df["value"], x=df["person_count"],
        orientation="h",
        marker_color=colors,
        text=[f"n={int(v):,}  ({p:.1f}%)" for v, p in zip(df["person_count"], df["pct_by_person"])],
        textposition="inside",
        insidetextanchor="middle",
    ))
    fig.update_layout(
        title=dict(text=(
            "Gender distribution of talk show guests (by unique person)"
            f"<br><sub>n={total:,} canonical guests · all shows combined · Wikidata P21</sub>"
        )),
        xaxis_title="Number of persons",
        yaxis_title="",
        width=900, height=max(400, 40 * len(df) + 120),
        showlegend=False,
    )
    fig = apply_font(fig)
    save_fig(fig, "f1_gender_distribution")
    fig.show()


In [ ]:
# F1b — Occupation distribution (top 30)
TOP_N_OCC = 30
if not occ_dist.empty:
    df = occ_dist.nlargest(TOP_N_OCC, "person_count").copy()
    # Unknown/Other last
    unknowns = df[df["occ"].isin(["(unknown)", ""])].copy()
    known    = df[~df["occ"].isin(["(unknown)", ""])].copy()
    df = pd.concat([known.sort_values("person_count"), unknowns], ignore_index=True)

    color_map = palette_for(df["occ"].tolist())
    colors = [color_map[v] for v in df["occ"]]
    total  = int(occ_dist["person_count"].sum())

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df["occ"], x=df["person_count"],
        orientation="h",
        marker_color=colors,
        text=[f"n={int(v):,}  ({p:.1f}%)" for v, p in zip(df["person_count"], df["pct_by_person"])],
        textposition="outside",
    ))
    fig.update_layout(
        title=dict(text=(
            f"Top {TOP_N_OCC} occupations — talk show guests (by unique person)"
            f"<br><sub>n={total:,} occupation assignments · all shows combined · Wikidata P106</sub>"
        )),
        xaxis_title="Number of persons",
        yaxis_title="",
        width=1000, height=max(400, 30 * len(df) + 150),
        showlegend=False,
        margin=dict(l=200),
    )
    fig = apply_font(fig)
    save_fig(fig, "f1_occupation_distribution")
    fig.show()


In [ ]:
# F1c — Party distribution (top 20)
TOP_N_PTY = 20
if not party_dist.empty:
    df = party_dist.nlargest(TOP_N_PTY, "person_count").copy()
    unknowns = df[df["party"].isin(["(unknown)", ""])].copy()
    known    = df[~df["party"].isin(["(unknown)", ""])].copy()
    df = pd.concat([known.sort_values("person_count"), unknowns], ignore_index=True)

    color_map = palette_for(df["party"].tolist())
    colors = [color_map[v] for v in df["party"]]
    total  = int(party_dist["person_count"].sum())

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df["party"], x=df["person_count"],
        orientation="h",
        marker_color=colors,
        text=[f"n={int(v):,}  ({p:.1f}%)" for v, p in zip(df["person_count"], df["pct_by_person"])],
        textposition="outside",
    ))
    fig.update_layout(
        title=dict(text=(
            f"Top {TOP_N_PTY} party affiliations — talk show guests (by unique person)"
            f"<br><sub>n={total:,} party assignments · all shows combined · Wikidata P102</sub>"
        )),
        xaxis_title="Number of persons",
        yaxis_title="",
        width=1000, height=max(400, 30 * len(df) + 150),
        showlegend=False,
        margin=dict(l=280),
    )
    fig = apply_font(fig)
    save_fig(fig, "f1_party_distribution")
    fig.show()


## F2 — Property Distribution Over Time (Line Charts)

Gender distribution per premiere year.
X-axis: year (Type D). Y-axis: person count or percentage.


In [ ]:
# F2 — Gender over time
if not gender_time.empty:
    df = gender_time.copy()
    df["premiere_year"] = pd.to_numeric(df["premiere_year"], errors="coerce")
    df = df.dropna(subset=["premiere_year"])
    df["premiere_year"] = df["premiere_year"].astype(int)
    df["person_count"]  = pd.to_numeric(df["person_count"], errors="coerce").fillna(0)

    # Compute pct within year
    year_totals = df.groupby("premiere_year")["person_count"].sum().rename("year_total")
    df = df.merge(year_totals, on="premiere_year")
    df["pct"] = (df["person_count"] / df["year_total"].replace(0, 1) * 100).round(1)

    genders = [g for g in df["gender"].unique() if g not in ("(unknown)", "")]
    unknowns = [g for g in df["gender"].unique() if g in ("(unknown)", "")]

    fig = go.Figure()
    for g in genders + unknowns:
        sub = df[df["gender"] == g].sort_values("premiere_year")
        color = GENDER_COLOR.get(g, PALETTE["gray"])
        fig.add_trace(go.Scatter(
            x=sub["premiere_year"], y=sub["pct"],
            mode="lines+markers", name=g,
            line=dict(color=color, width=2),
            marker=dict(size=5),
            hovertemplate="%{x}: %{y:.1f}% (%{customdata} persons)<extra></extra>",
            customdata=sub["person_count"].astype(int),
        ))
    fig.update_layout(
        title=dict(text=(
            "Gender distribution of talk show guests over time"
            "<br><sub>Percentage of annual unique guest appearances · all shows · Wikidata P21</sub>"
        )),
        xaxis_title="Year",
        yaxis_title="Percentage (%)",
        width=1000, height=500,
        legend_title="Gender",
    )
    fig = apply_font(fig)
    save_fig(fig, "f2_gender_over_time")
    fig.show()


## F3 — Cross-Tabulation Charts

Binary cross-tab (gender × X): centered stacked bar chart (female left, male right).
Non-binary cross-tab (occupation × party): grouped bar chart.

Source: `gender_by_occupation.csv`, `gender_by_party.csv`, `party_by_occupation.csv`.


In [ ]:
# F3a — Gender × Occupation (centered stacked bar)
# Left (female) in blue, right (male) in orange; unknown in gray.
TOP_N_OCC_F3 = 20

def make_centered_stacked_bar(df_pivot, title, subtitle, left_col, right_col,
                               left_label, right_label, fname, top_n=None):
    df = df_pivot.copy()
    numeric_cols = [c for c in df.columns if c != df.columns[0]]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    if left_col not in df.columns or right_col not in df.columns:
        print(f"  Skipped {fname}: columns not found. Available: {list(df.columns)}")
        return

    unknown_cols = [c for c in numeric_cols if c not in (left_col, right_col)]
    df["_total"] = df[numeric_cols].sum(axis=1)
    df["_left_pct"]  = df[left_col]  / df["_total"].replace(0, 1) * 100
    df["_right_pct"] = df[right_col] / df["_total"].replace(0, 1) * 100
    df["_unk_pct"]   = df[unknown_cols].sum(axis=1) / df["_total"].replace(0, 1) * 100

    cat_col = df.columns[0]
    if top_n:
        df = df.nlargest(top_n, "_total")
    df = df.sort_values("_left_pct")  # ascending so top of chart = highest left share

    n_total = int(df["_total"].sum())

    def _annot(pct):
        return [f"{p:.0f}%" if p >= 5 else "" for p in pct]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=left_label, y=df[cat_col], x=-df["_left_pct"],
        orientation="h", marker_color=PALETTE["blue"],
        text=_annot(df["_left_pct"]), textposition="inside", insidetextanchor="middle",
    ))
    if unknown_cols:
        fig.add_trace(go.Bar(
            name="(unknown)", y=df[cat_col], x=-df["_unk_pct"] / 2,
            orientation="h", marker_color=PALETTE["gray"],
            text=_annot(df["_unk_pct"]), textposition="inside", insidetextanchor="middle",
            showlegend=True,
        ))
    fig.add_trace(go.Bar(
        name=right_label, y=df[cat_col], x=df["_right_pct"],
        orientation="h", marker_color=PALETTE["orange"],
        text=_annot(df["_right_pct"]), textposition="inside", insidetextanchor="middle",
    ))
    # Add zero line
    fig.add_vline(x=0, line_width=2, line_color="black")
    fig.update_layout(
        barmode="relative",
        title=dict(text=f"{title}<br><sub>{subtitle} · n={n_total:,}</sub>"),
        xaxis=dict(title=f"← {left_label}  |  {right_label} →",
                   tickvals=[-75,-50,-25,0,25,50,75],
                   ticktext=["75%","50%","25%","0","25%","50%","75%"]),
        yaxis_title="",
        width=1000, height=max(400, 30 * len(df) + 150),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    return apply_font(fig)

if not gender_by_occ.empty:
    occ_col = gender_by_occ.columns[0]
    # Detect column names (may be German or English depending on Wikidata labels)
    gcols = [c for c in gender_by_occ.columns if c != occ_col]
    left_c  = next((c for c in gcols if "weiblich" in c.lower() or "female" in c.lower()), gcols[0] if gcols else None)
    right_c = next((c for c in gcols if "männlich" in c.lower() or "male" in c.lower()), gcols[1] if len(gcols) > 1 else None)
    if left_c and right_c:
        fig = make_centered_stacked_bar(
            gender_by_occ, title="Gender distribution by occupation",
            subtitle=f"top {TOP_N_OCC_F3} occupations · all shows · Wikidata P21+P106",
            left_col=left_c, right_col=right_c,
            left_label=left_c, right_label=right_c,
            fname="f3_gender_by_occupation", top_n=TOP_N_OCC_F3,
        )
        if fig:
            save_fig(fig, "f3_gender_by_occupation")
            fig.show()
    else:
        print(f"Could not identify left/right gender columns in {list(gender_by_occ.columns)}")


In [ ]:
# F3b — Gender × Party (centered stacked bar)
TOP_N_PTY_F3 = 15
if not gender_by_party.empty:
    pty_col = gender_by_party.columns[0]
    gcols = [c for c in gender_by_party.columns if c != pty_col]
    left_c  = next((c for c in gcols if "weiblich" in c.lower() or "female" in c.lower()), gcols[0] if gcols else None)
    right_c = next((c for c in gcols if "männlich" in c.lower() or "male" in c.lower()), gcols[1] if len(gcols) > 1 else None)
    if left_c and right_c:
        fig = make_centered_stacked_bar(
            gender_by_party, title="Gender distribution by party affiliation",
            subtitle=f"top {TOP_N_PTY_F3} parties · all shows · Wikidata P21+P102",
            left_col=left_c, right_col=right_c,
            left_label=left_c, right_label=right_c,
            fname="f3_gender_by_party", top_n=TOP_N_PTY_F3,
        )
        if fig:
            save_fig(fig, "f3_gender_by_party")
            fig.show()


In [ ]:
# F3c — Occupation × Party (grouped bar chart)
TOP_N_OCC_F3C = 15
TOP_N_PTY_F3C = 8
if not party_by_occ.empty:
    occ_col = party_by_occ.columns[0]
    party_cols = [c for c in party_by_occ.columns if c != occ_col]

    df = party_by_occ.copy()
    for c in party_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["_total"] = df[party_cols].sum(axis=1)
    df = df.nlargest(TOP_N_OCC_F3C, "_total")

    # Limit parties to top N by column sum
    pty_sums = {c: df[c].sum() for c in party_cols}
    top_parties = sorted(pty_sums, key=pty_sums.get, reverse=True)[:TOP_N_PTY_F3C]
    color_map = palette_for(top_parties)
    n_total = int(df["_total"].sum())

    fig = go.Figure()
    for i, pty in enumerate(top_parties):
        fig.add_trace(go.Bar(
            name=pty, x=df[occ_col], y=df[pty],
            marker_color=color_map[pty],
        ))
    fig.update_layout(
        barmode="group",
        title=dict(text=(
            f"Party affiliation by occupation (top {TOP_N_OCC_F3C} occ × top {TOP_N_PTY_F3C} parties)"
            f"<br><sub>n={n_total:,} · all shows · Wikidata P106+P102</sub>"
        )),
        xaxis_title="Occupation",
        yaxis_title="Number of persons",
        xaxis_tickangle=-35,
        width=1200, height=600,
        legend_title="Party",
    )
    fig = apply_font(fig)
    save_fig(fig, "f3_party_by_occupation")
    fig.show()


## F4 — Continuous Distribution (Age)

Histogram of appearance ages (premiere_year − birth_year), binned in 10-year intervals.
Violin plot alongside for richer summary.
Source: `age_distribution.csv`.


In [ ]:
# F4 — Age distribution (histogram + stats)
if not age_dist.empty and "age_bin" in age_dist.columns:
    df = age_dist.copy()
    df = df[df["age_bin"] != ""]

    n_appearances = int(df["appearance_count"].sum())
    n_persons     = int(df["unique_persons"].sum())

    # Histogram from binned data (bar chart since bins are pre-computed)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df["age_bin"], y=df["appearance_count"],
        name="Appearances",
        marker_color=PALETTE["blue"],
        text=df["appearance_count"].astype(int),
        textposition="outside",
    ))
    fig.add_trace(go.Bar(
        x=df["age_bin"], y=df["unique_persons"],
        name="Unique persons",
        marker_color=PALETTE["orange"],
        text=df["unique_persons"].astype(int),
        textposition="outside",
    ))
    fig.update_layout(
        barmode="group",
        title=dict(text=(
            "Age distribution at time of appearance"
            f"<br><sub>n={n_appearances:,} appearances · {n_persons:,} unique persons"
            " · year-of-birth precision only · Wikidata P569</sub>"
        )),
        xaxis_title="Age at appearance (years)",
        yaxis_title="Count",
        width=1000, height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig = apply_font(fig)
    save_fig(fig, "f4_age_distribution")
    fig.show()
else:
    print("age_distribution.csv empty or missing age_bin column — run 50_analysis.ipynb first")


## F5 — Hierarchy Visualization (TASK-A02 — Deferred)

**Status: Blocked** — requires class hierarchy to be fully resolved (P279 walk for all occupation QIDs, including Q488205 and similar). See TASK-A02 in `open-tasks.md`.

Known issues with the current class hierarchy:
- Some occupation QIDs (e.g. Q488205 Schauspieler) are not yet `class_hierarchy_resolved`
- The current sunburst shows "Person" as center and each top-level class as its own child — broken hierarchy
- Fix requires: `entity_access.all_outlink_fetch` + Phase 2 class hierarchy tools for all occupation QIDs

**Do not finalize this cell until TASK-A02 prerequisites are met.**


In [ ]:
# F5 — Occupation hierarchy (Sunburst + Sankey)
# BLOCKED: see TASK-A02 — class hierarchy not fully resolved
# Prerequisites:
#   1. all occupation QIDs are class_hierarchy_resolved (including Q488205)
#   2. class_resolution_map.csv is complete
#   3. TASK-A06 full-fetch has been run and cache is populated
#
# Once unblocked, implement here following:
#   - visualization-principles.md §Sunburst / Sankey
#   - Multi-parent strategy: primary-parent assignment (document in cell)
#   - 5% Other cutoff on combined diagram
#   - One combined + one per-show sunburst (TASK-A02 §Required outputs)
#   - Export PNG + PDF + HTML

crm_path = ANALYSIS_DIR.parent.parent / "20_candidate_generation/wikidata/projections/class_resolution_map.csv"
if crm_path.exists() and not occ_dist.empty:
    crm = pd.read_csv(crm_path, dtype=str).fillna("")
    resolved_pct = (crm["core_class_qid"] != "").mean() * 100 if "core_class_qid" in crm.columns else 0
    print(f"class_resolution_map: {len(crm):,} rows, {resolved_pct:.1f}% resolved")
    print("F5 sunburst: BLOCKED — verify TASK-A02 prerequisites before implementing")
else:
    print("class_resolution_map.csv not found — TASK-A02 blocked")


## Page Rank — Node Graph (TASK-A03 — Deferred)

**Status: Open** — requires page rank scores. See TASK-A03 in `open-tasks.md`.

Required outputs (per TASK-A03):
1. Node graph for all person instances (nodes sized/colored by rank score)
2. Node graph for all classes
3. Combined view

Implementation notes (per visualization-principles.md §Node Graphs):
- Node size proportional to rank/importance score
- Label only top-N nodes (configurable constant `LABEL_TOP_N = 20`)
- Edges show only when edge weight exceeds configurable threshold
- Export at scale=4 for label readability


In [ ]:
# Page rank node graph
# BLOCKED: see TASK-A03 — page rank data needed
# Check for legacy pagerank output from Phase 4 analysis
pagerank_paths = [
    Path("data/40_analysis/pagerank_persons.csv"),
    Path("data/50_analysis/all/pagerank_persons.csv"),
]
pr_path = next((p for p in pagerank_paths if p.exists()), None)
if pr_path:
    pr = pd.read_csv(pr_path, dtype=str).fillna("")
    pr["score"] = pd.to_numeric(pr.get("pagerank", pr.get("score", pd.Series())), errors="coerce").fillna(0)
    print(f"Page rank data found: {pr_path}  ({len(pr):,} rows)")
    print("TODO: implement node graph visualization once TASK-A03 is active")
    print(f"  Top 5 by rank: {pr.nlargest(5, 'score')[['canonical_label','score']].to_string(index=False)}")
else:
    print("No page rank CSV found — TASK-A03 requires page rank computation first")
    print("Expected: data/50_analysis/all/pagerank_persons.csv")
